# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta_obj = dataset.metadata
meta_json = dataset.metadata.to_json()
print(f"{meta_json['name']}: {meta_json['description']}")

## 2. Data Overview
Review available record sets, fields, columns and their IDs. All are referenced by their `@id` fields.

Let's list all the record sets (`cr:RecordSet`) in the dataset, then for each record set, print its fields and field `@id`s.

In [ ]:
# Gather record sets by @id
record_sets = [rs for rs in meta_obj.record_sets]
print(f"Available Record Sets (@id): {[rs.id for rs in record_sets]}")

for rs in record_sets:
    print(f"\nRecord Set: {rs.id}")
    # List its fields
    if hasattr(rs, 'fields'):
        print(" Fields and their IDs:")
        for f in rs.fields:
            print(f"  - {f.name}: {f.id}")
        # List underlying columns too
        print(" Columns and their IDs:")
        # For each field, if it has a column
        for f in rs.fields:
            col = getattr(f, 'column', None)
            if col is not None:
                print(f"  - {getattr(col, 'name', col.id)}: {col.id}")
    else:
        print(" (No fields found)")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

For demonstration, let's extract the first record set.

In [ ]:
# Extract data from each record set
# List record set @id's
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for rset_id in record_set_ids:
    records = list(dataset.records(record_set=rset_id))
    if len(records) > 0:
        dataframes[rset_id] = pd.DataFrame(records)
    else:
        dataframes[rset_id] = pd.DataFrame()

# Show the columns for the first populated record set
for rset_id in record_set_ids:
    if not dataframes[rset_id].empty:
        print(f"Columns for '{rset_id}':", dataframes[rset_id].columns.tolist())
        display(dataframes[rset_id].head())
        break

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section will:
- Select a numeric field for thresholding and normalization,
- Perform basic grouping, filtering, and normalization.

Adjust variables `selected_record_set`, `numeric_field_id`, and `group_field_id` below based on the overview and record set you wish to explore.

In [ ]:
# Pick the first non-empty record set
selected_record_set = None
for rset_id, df in dataframes.items():
    if not df.empty:
        selected_record_set = rset_id
        break

# For demonstration, guess a numeric field (adjust if needed): e.g. 'Molecular Weight', 'Abundance', etc.
sample_df = dataframes[selected_record_set]
numeric_candidates = [c for c in sample_df.columns if sample_df[c].dtype in ['float64', 'int64']]
print('Numeric field candidates:', numeric_candidates)
numeric_field_id = numeric_candidates[0] if numeric_candidates else sample_df.columns[-1]

# Group field (use a field with categorical/limited values, e.g. 'Sample', 'Protein', etc.)
group_field_candidates = [c for c in sample_df.columns if sample_df[c].dtype == 'object' and c != numeric_field_id]
group_field_id = group_field_candidates[0] if group_field_candidates else None

threshold = sample_df[numeric_field_id].median() if numeric_field_id in sample_df.columns else 0

filtered_df = sample_df[sample_df[numeric_field_id] > threshold] if numeric_field_id in sample_df.columns else sample_df
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

if numeric_field_id in filtered_df.columns:
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

if group_field_id:
    # Only show mean for numeric columns
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id}: (showing means)")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
if numeric_field_id in sample_df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(sample_df[numeric_field_id], kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

# If a group field exists, plot the group means of the normalized field
if group_field_id and f"{numeric_field_id}_normalized" in filtered_df.columns:
    group_means = filtered_df.groupby(group_field_id)[f"{numeric_field_id}_normalized"].mean().reset_index()
    plt.figure(figsize=(8, 4))
    sns.barplot(data=group_means, x=group_field_id, y=f"{numeric_field_id}_normalized")
    plt.ylabel(f"Mean of {numeric_field_id} (normalized)")
    plt.xticks(rotation=45)
    plt.title(f"Mean normalized {numeric_field_id} by {group_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated loading a mass spectrometry protein analysis dataset via the Croissant schema and `mlcroissant`, exploring its structure, extracting tabular data by record set and `@id`, running basic EDA and performing simple visualizations.

Key steps included:
- Listing all record sets, fields, and their `@id` references,
- Programmatic, flexible loading of any record set (referenced by `@id`),
- Filtering and normalization of numerical fields, and grouping by categorical fields as part of an initial EDA pipeline.

You can adapt field and group variable selections to focus on specific aspects of your data, or extend the EDA and visualization steps for domain analysis.